
# Predictable Profits? A Machine Learning Analysis of US IPO Underpricing (2019-2024)

**Student Name:** [Your Name]  
**Course:** Business Analytics Term Project  
**Date:** April 2026

---

## 1. Executive Summary
This project investigates whether first-day IPO "underpricing" (the percentage change from offer price to first-day close) can be predicted using a combination of traditional financial metrics, market sentiment, and textual analysis of SEC S-1 prospectuses.

### Research Question
> *Can we predict first-day IPO underpricing in the US stock market using traditional deal features combined with textual sentiment and readability features extracted from company prospectuses?*

### Methodology
1. **Data Collection:** Scraped ~1,700 IPOs from 2019-2024 via StockAnalysis.com and SEC EDGAR.
2. **Feature Engineering:** Extracted Loughran-McDonald sentiment ratios, Gunning-Fog readability, and Hanley-Hoberg prospectus uniqueness.
3. **Hypothesis Testing:** Conducted 6 statistical tests to validate drivers of underpricing.
4. **Machine Learning:** Trained a Gradient Boosted Decision Tree (LightGBM) model to predict underpricing.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Custom modules
import sys
sys.path.append('..')
from src import hypothesis_tests as ht

# Styling
plt.style.use('ggplot')
sns.set_palette('viridis')
%matplotlib inline



## 2. Data Loading & Overview
We load the processed dataset which includes ~1,600 IPOs and ~200 textual features.


In [ ]:

df = pd.read_parquet("../data/processed/ipo_features.parquet")
print(f"Dataset Shape: {df.shape}")
df.head()



## 3. Exploratory Data Analysis (EDA)
### 3.1 Distribution of Underpricing
First-day returns are notoriously skewed. We use "winsorized" underpricing to handle extreme outliers.


In [ ]:

plt.figure(figsize=(10, 6))
sns.histplot(df['underpricing'], bins=50, kde=True, color='teal')
plt.title('Distribution of First-Day IPO Underpricing')
plt.xlabel('Underpricing (%)')
plt.ylabel('Frequency')
plt.show()

print(f"Mean Underpricing: {df['underpricing'].mean():.2%}")
print(f"Median Underpricing: {df['underpricing'].median():.2%}")



## 4. Hypothesis Testing
We test six core hypotheses to understand the drivers of underpricing.


In [ ]:

# H1: Tech vs Non-Tech
res_h1 = ht.test_h1_tech_vs_nontech(df)
ht.report(res_h1)

# H3: Sentiment vs Underpricing
df_text = df[df['lm_negative_ratio'].notna()]
if not df_text.empty:
    res_h3 = ht.test_h3_lm_negative_underpricing(df_text)
    ht.report(res_h3)



## 5. Machine Learning Modeling
We use a LightGBM model to predict underpricing. We evaluate performance using MAE and R-squared.


In [ ]:

from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Define features
features = [
    'ipo_year', 'ipo_month', 'ipo_dayofweek', 'is_quarter_end_month',
    'vix_at_pricing', 'nasdaq_30d_return', 'nasdaq_30d_volatility',
    'offer_price', 'lm_negative_ratio', 'gunning_fog', 'prospectus_uniqueness'
]
# Only use rows where we have text features for a 'full' model comparison
df_ml = df.dropna(subset=['lm_negative_ratio', 'underpricing'])

# Impute missing values for ML
X = df_ml[features].fillna(df_ml[features].median())
y = df_ml['underpricing']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LGBMRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")



### 5.1 Feature Importance (SHAP)
SHAP values tell us which features contributed most to the model's predictions.


In [ ]:

import shap
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)
